# Defect Detection in Hot Rolling - Leak-Free, Regularized Ensemble Pipeline

**Objective:** detect Alpha defects with a safety-first model. The business acceptance rule is stricter than ordinary classification: **zero false negatives** is the first priority, and false positives must be controlled as much as possible.

This notebook implements an industry-grade ML pipeline that resolves key modeling issues:
1. **Absolutely Zero Data Leakage**: All preprocessing (missing value median imputation, robust scaling, feature selection) is performed strictly inside the cross-validation loop.
2. **Overfitting Prevention**: Tree-based models are highly regularized (using shallow tree depth, row/column subsampling, and L1/L2 penalties) to prevent training-set memorization and ensure generalization.
3. **Rank-Normalized Ensembling**: Predictions from Logistic Regression, XGBoost, CatBoost, and HistGradientBoosting are blended using fold-wise ranks (percentiles) to align probability scales.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_score, recall_score,
    f1_score, average_precision_score, roc_auc_score, precision_recall_curve,
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from scipy.stats import rankdata

RANDOM_STATE = 42
N_SPLITS = 5
PROJECT_DIR = Path(r"D:\Adarsh\TataSteelAIHackathon")
DATA_DIR = PROJECT_DIR / "Data"
SUBMISSION_DIR = PROJECT_DIR / "submissions_v5"
SUBMISSION_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"

## 1. Load Data and Check Schema

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

assert train.shape == (1352, 51), "Unexpected train shape"
assert test.shape == (339, 50), "Unexpected test shape"
assert {"CoilID", "Y"}.issubset(train.columns)
assert "CoilID" in test.columns
train.head()

## 2. Target Class Distribution and Missingness Summary

In [ ]:
target_counts = train["Y"].astype(int).value_counts().sort_index()
target_rate = train["Y"].mean()
missing_summary = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_pct": train.isna().mean() * 100,
}).sort_values("missing_count", ascending=False)

print("Target distribution:")
print(target_counts)
print(f"Defect rate: {target_rate:.2%}")

print("\nTop missing columns:")
display(missing_summary.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(x=target_counts.index.astype(str), y=target_counts.values, ax=axes[0])
axes[0].set_title("Class Balance")
axes[0].set_xlabel("Y: 0 = No Defect, 1 = Defect")
axes[0].set_ylabel("Coil Count")
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, f"{v:,}", ha="center", weight="bold")
miss = missing_summary[missing_summary["missing_count"] > 0].head(15)
sns.barplot(data=miss.reset_index(), y="index", x="missing_pct", ax=axes[1], color="#4C78A8")
axes[1].set_title("Missingness by Feature")
axes[1].set_xlabel("Missing %")
axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

## 3. Row-Wise Feature Engineering

We construct engineered features representing statistical summaries of process sensors. Because they only compute features row-by-row (using properties of a single coil's parameters), they do not leak information across samples.

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()
    drop_cols = ["CoilID", "coilid", "Y", "y"]
    X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors="ignore").copy()
    x_cols = [c for c in X.columns if c.upper().startswith("X")]
    
    for col in x_cols:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        
    base = X[x_cols]
    
    # Global row aggregates
    X["row_mean"] = base.mean(axis=1)
    X["row_std"] = base.std(axis=1).fillna(0)
    X["row_min"] = base.min(axis=1)
    X["row_max"] = base.max(axis=1)
    X["row_range"] = X["row_max"] - X["row_min"]
    X["row_median"] = base.median(axis=1)
    X["row_iqr"] = base.quantile(0.75, axis=1) - base.quantile(0.25, axis=1)
    X["row_missing"] = base.isna().sum(axis=1)
    X["row_skew"] = base.skew(axis=1).fillna(0)
    X["row_kurt"] = base.kurt(axis=1).fillna(0)

    # Adjacent process differences
    values = base.to_numpy(dtype=float)
    diffs = np.diff(values, axis=1)
    X["adj_diff_mean"] = np.nanmean(diffs, axis=1)
    X["adj_diff_std"] = np.nanstd(diffs, axis=1)
    X["adj_diff_abs_mean"] = np.nanmean(np.abs(diffs), axis=1)
    X["adj_diff_max"] = np.nanmax(diffs, axis=1)
    X["adj_diff_min"] = np.nanmin(diffs, axis=1)

    # Process stage averages
    stages = {
        "s1": [f"X{i}" for i in range(1, 17)],
        "s2": [f"X{i}" for i in range(17, 34)],
        "s3": [f"X{i}" for i in range(34, 50)],
    }
    for stage, cols in stages.items():
        s = X[cols]
        X[f"{stage}_mean"] = s.mean(axis=1)
        X[f"{stage}_std"] = s.std(axis=1).fillna(0)
        X[f"{stage}_min"] = s.min(axis=1)
        X[f"{stage}_max"] = s.max(axis=1)
        X[f"{stage}_range"] = X[f"{stage}_max"] - X[f"{stage}_min"]
        X[f"{stage}_median"] = s.median(axis=1)

    eps = 1e-6
    X["s1_s2_mean_diff"] = X["s1_mean"] - X["s2_mean"]
    X["s2_s3_mean_diff"] = X["s2_mean"] - X["s3_mean"]
    X["s1_s3_mean_diff"] = X["s1_mean"] - X["s3_mean"]
    X["s1_s2_ratio"] = X["s1_mean"] / (X["s2_mean"].abs() + eps)
    X["s2_s3_ratio"] = X["s2_mean"] / (X["s3_mean"].abs() + eps)
    X["s1_s3_ratio"] = X["s1_mean"] / (X["s3_mean"].abs() + eps)

    for a, b in [("X13", "X36"), ("X10", "X36"), ("X13", "X41"), ("X30", "X36"), ("X32", "X39")]:
        X[f"{a}_minus_{b}"] = X[a] - X[b]
        X[f"{a}_ratio_{b}"] = X[a] / (X[b].abs() + eps)

    return X

X_train_full = build_features(train)
X_test_full = build_features(test)
y = train['Y'].astype(int)

X_train_full, X_test_full = X_train_full.align(X_test_full, join='left', axis=1, fill_value=0.0)
x_cols = [c for c in X_train_full.columns if c.upper().startswith("X") and "_" not in c]

print("Training matrix shape:", X_train_full.shape)
print("Columns aligned:", list(X_train_full.columns) == list(X_test_full.columns))

## 4. Leak-Free Cross-Validation and Hyperparameter Tuning

We define the 5-fold Stratified K-Fold. Inside the loop, we perform median imputation, robust scaling, and feature selection (for LR only) fit ONLY on the training fold, which eliminates data leakage. 

We use highly regularized configurations of XGBoost, CatBoost, and HistGradientBoosting to prevent overfitting.

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
class_ratio = (y == 0).sum() / (y == 1).sum()

models = {
    'lr_l2': LogisticRegression(class_weight="balanced", penalty="l2", C=0.1, random_state=RANDOM_STATE, max_iter=1000),
    'xgb_reg': XGBClassifier(
        max_depth=4, learning_rate=0.01, n_estimators=300,
        reg_alpha=10, reg_lambda=50, subsample=0.6, colsample_bytree=0.6,
        scale_pos_weight=class_ratio, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'cat_reg': CatBoostClassifier(
        depth=5, learning_rate=0.01, iterations=600,
        l2_leaf_reg=50, loss_function="Logloss",
        class_weights=[1, class_ratio], random_seed=RANDOM_STATE,
        verbose=False, allow_writing_files=False
    ),
    'hgb_reg': HistGradientBoostingClassifier(
        max_depth=2, learning_rate=0.02, max_iter=150, l2_regularization=50,
        class_weight="balanced", random_state=RANDOM_STATE
    )
}

oof_ranks = {name: np.zeros(len(y)) for name in models}
train_aps = {name: [] for name in models}
val_aps = {name: [] for name in models}

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train_full, y), start=1):
    X_tr, y_tr = X_train_full.iloc[tr_idx].copy(), y.iloc[tr_idx].copy()
    X_val, y_val = X_train_full.iloc[val_idx].copy(), y.iloc[val_idx].copy()
    
    # Missing indicators
    for col in x_cols:
        if X_tr[col].isna().sum() > 0:
            X_tr[f"{col}_is_missing"] = X_tr[col].isna().astype(int)
            X_val[f"{col}_is_missing"] = X_val[col].isna().astype(int)
            
    all_tr_cols = X_tr.columns
    X_val = X_val.reindex(columns=all_tr_cols, fill_value=0)
    
    # Impute missing values with median
    imputer = SimpleImputer(strategy='median')
    X_tr[x_cols] = imputer.fit_transform(X_tr[x_cols])
    X_val[x_cols] = imputer.transform(X_val[x_cols])
    
    X_tr_clean = X_tr.fillna(0).replace([np.inf, -np.inf], 0)
    X_val_clean = X_val.fillna(0).replace([np.inf, -np.inf], 0)
    
    # Robust scaling
    scaler = RobustScaler()
    X_tr_scaled = pd.DataFrame(scaler.fit_transform(X_tr_clean), columns=all_tr_cols)
    X_val_scaled = pd.DataFrame(scaler.transform(X_val_clean), columns=all_tr_cols)
    
    for name, model in models.items():
        import copy
        m = copy.deepcopy(model)
        
        if name == 'lr_l2':
            selector = SelectKBest(score_func=f_classif, k=30)
            selector.fit(X_tr_scaled, y_tr)
            sel_cols = all_tr_cols[selector.get_support()]
            m.fit(X_tr_scaled[sel_cols], y_tr)
            tr_p = m.predict_proba(X_tr_scaled[sel_cols])[:, 1]
            val_p = m.predict_proba(X_val_scaled[sel_cols])[:, 1]
        else:
            m.fit(X_tr_scaled, y_tr)
            tr_p = m.predict_proba(X_tr_scaled)[:, 1]
            val_p = m.predict_proba(X_val_scaled)[:, 1]
            
        train_aps[name].append(average_precision_score(y_tr, tr_p))
        val_aps[name].append(average_precision_score(y_val, val_p))
        
        # Rank-normalize validation probabilities
        oof_ranks[name][val_idx] = rankdata(val_p) / len(val_p)

## 5. Model Performance Dashboard

In [ ]:
print("--- MODEL DIAGNOSTICS ---")
for name in models:
    mean_tr_ap = np.mean(train_aps[name])
    mean_val_ap = np.mean(val_aps[name])
    overall_ap_rank = average_precision_score(y, oof_ranks[name])
    overall_auc_rank = roc_auc_score(y, oof_ranks[name])
    print(f"{name}:")
    print(f"  Training AP (Mean): {mean_tr_ap:.4f} | Validation AP (Mean): {mean_val_ap:.4f}")
    print(f"  OOF AP (Ranked): {overall_ap_rank:.4f} | OOF AUC (Ranked): {overall_auc_rank:.4f}")

print("\n--- ENSEMBLE BLENDING ---")
blend_rank = 0.20 * oof_ranks['hgb_reg'] + 0.20 * oof_ranks['xgb_reg'] + 0.60 * oof_ranks['cat_reg']
blend_ap = average_precision_score(y, blend_rank)
blend_auc = roc_auc_score(y, blend_rank)
print(f"Ensemble AP: {blend_ap:.4f} | Ensemble AUC: {blend_auc:.4f}")

## 6. Threshold Optimization and Evaluation

In [ ]:
# 1. Best F1 threshold
best_f1 = 0
best_thr = 0
best_rec = 0
best_prec = 0
thresholds = np.linspace(0, 1, 1000)
for thr in thresholds:
    preds = (blend_rank >= thr).astype(int)
    f1 = f1_score(y, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr
        best_rec = recall_score(y, preds, zero_division=0)
        best_prec = precision_score(y, preds, zero_division=0)

best_preds = (blend_rank >= best_thr).astype(int)
best_k = int(best_preds.sum())
tn, fp, fn, tp = confusion_matrix(y, best_preds).ravel()

print(f"Best F1: {best_f1:.4f} at Rank Threshold {best_thr:.4f}")
print(f"  Recall: {best_rec:.2%} | Precision: {best_prec:.2%}")
print(f"  TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn} | Predicted Positives: {best_k}")

# 2. Zero FN threshold
pos_ranks = blend_rank[y == 1]
zero_fn_thr = float(np.min(pos_ranks))
zero_fn_preds = (blend_rank >= zero_fn_thr).astype(int)
zero_fn_k = int(zero_fn_preds.sum())
z_tn, z_fp, z_fn, z_tp = confusion_matrix(y, zero_fn_preds).ravel()

print(f"\nZero FN Threshold: {zero_fn_thr:.6f}")
print(f"  Recall: {recall_score(y, zero_fn_preds):.2%} | Precision: {precision_score(y, zero_fn_preds, zero_division=0):.2%}")
print(f"  TP: {z_tp} | FP: {z_fp} | FN: {z_fn} | TN: {z_tn} | Predicted Positives: {zero_fn_k}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
cm = confusion_matrix(y, best_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=["Pred 0", "Pred 1"], yticklabels=["Actual 0", "Actual 1"], ax=axes[0])
axes[0].set_title("Confusion Matrix - Best F1 Ensemble")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

precision, recall, _ = precision_recall_curve(y, blend_rank)
axes[1].plot(recall, precision, linewidth=2.5, color="#1B9E77")
axes[1].axhline(0.90, color="red", linestyle="--", label="90% Precision Target")
axes[1].axvline(1.00, color="gray", linestyle=":", label="100% Recall Target")
axes[1].scatter([best_rec], [best_prec], color="black", s=80, zorder=5, label="Selected Best F1")
axes[1].set_title("Precision-Recall Curve - Ensemble")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_xlim(0, 1.02); axes[1].set_ylim(0, 1.02); axes[1].legend(loc="best")
plt.tight_layout(); plt.show()

## 7. Final Training and Test Submission Generation

We train the regularized models on the full training dataset using the complete scalers and imputers, rank-normalize the test predictions, blend them, and write out all threshold variants.

In [ ]:
X_tr_full = X_train_full.copy()
X_te_full = X_test_full.copy()

# Missing indicators
for col in x_cols:
    if X_tr_full[col].isna().sum() > 0:
        X_tr_full[f"{col}_is_missing"] = X_tr_full[col].isna().astype(int)
        X_te_full[f"{col}_is_missing"] = X_te_full[col].isna().astype(int)

all_full_cols = X_tr_full.columns
X_te_full = X_te_full.reindex(columns=all_full_cols, fill_value=0)

# Impute full set
imputer_full = SimpleImputer(strategy='median')
X_tr_full[x_cols] = imputer_full.fit_transform(X_tr_full[x_cols])
X_te_full[x_cols] = imputer_full.transform(X_te_full[x_cols])

X_tr_full_clean = X_tr_full.fillna(0).replace([np.inf, -np.inf], 0)
X_te_full_clean = X_te_full.fillna(0).replace([np.inf, -np.inf], 0)

# Scale full set
scaler_full = RobustScaler()
X_tr_full_scaled = pd.DataFrame(scaler_full.fit_transform(X_tr_full_clean), columns=all_full_cols)
X_te_full_scaled = pd.DataFrame(scaler_full.transform(X_te_full_clean), columns=all_full_cols)

test_ranks = {}
for name, model in models.items():
    import copy
    m = copy.deepcopy(model)
    
    if name == 'lr_l2':
        selector = SelectKBest(score_func=f_classif, k=30)
        selector.fit(X_tr_full_scaled, y)
        sel_cols = all_full_cols[selector.get_support()]
        m.fit(X_tr_full_scaled[sel_cols], y)
        test_p = m.predict_proba(X_te_full_scaled[sel_cols])[:, 1]
    else:
        m.fit(X_tr_full_scaled, y)
        test_p = m.predict_proba(X_te_full_scaled)[:, 1]
        
    test_ranks[name] = rankdata(test_p) / len(test_p)

test_blend_rank = 0.20 * test_ranks['hgb_reg'] + 0.20 * test_ranks['xgb_reg'] + 0.60 * test_ranks['cat_reg']

# Generate submissions
def save_sub(preds, name):
    sub = pd.DataFrame({"CoilID": test["CoilID"], "Y": preds})
    path = SUBMISSION_DIR / name
    sub.to_csv(path, index=False)
    print(f"Saved: {name} (Predicted defects: {int(preds.sum())})")

save_sub((test_blend_rank >= zero_fn_thr).astype(int), "submission_recall_first_zero_fn.csv")
save_sub((test_blend_rank >= best_thr).astype(int), "submission_balanced_best_f1.csv")
for rate in [0.25, 0.15, 0.10, 0.05]:
    k = max(1, int(round(len(test) * rate)))
    thr = float(np.sort(test_blend_rank)[-k])
    save_sub((test_blend_rank >= thr).astype(int), f"submission_test_top_{int(rate*100)}pct_probe.csv")